In [1]:
import json
import logging
import pandas as pd
import numpy as np
import pickle
import time
import os
import uuid
import getpass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
import pubchempy as pcp
from tqdm.notebook import tqdm
import sys
import duckdb

sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.pubchem_retrieval as pcr

# Creator information
creator_name = getpass.getuser()
timestamp = datetime.now().isoformat()

In [2]:
# Required: Input file with experimental data (compound list for atlas)
ATLAS_INPUT = "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"

# Atlas Configuration
ATLAS_NAME = "HILIC Positive QC Default"  # User-defined atlas name
ATLAS_DESC = f"Default QC compounds for HILIC Positive as of {timestamp}"  # Optional description

# Required: Database file path
DATABASE_PATH = "/Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb"

## Load and Validate Input Data

In [3]:
# Load and validate input data
print(f"Loading atlas compound list from: {ATLAS_INPUT}")
delimiter = '\t' if ATLAS_INPUT.endswith(('.tsv', '.tab', '.txt')) else ','
atlas_compounds_df = pd.read_csv(ATLAS_INPUT, sep=delimiter)
print(f"Loaded {len(atlas_compounds_df)} compounds for atlas")

# Check for required columns
required_columns = ['inchi_key']
missing_columns = [col for col in required_columns if col not in atlas_compounds_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Get unique InChI keys for atlas
atlas_inchi_keys = atlas_compounds_df['inchi_key'].dropna().unique()
print(f"Unique compounds for atlas: {len(atlas_inchi_keys)}")

# Auto-detect chromatography and polarity from input data - REQUIRED
if 'chromatography' not in atlas_compounds_df.columns or atlas_compounds_df['chromatography'].isna().all():
    raise ValueError("Input data must contain 'chromatography' column with values")

if 'polarity' not in atlas_compounds_df.columns or atlas_compounds_df['polarity'].isna().all():
    raise ValueError("Input data must contain 'polarity' column with values")

ATLAS_CHROMATOGRAPHY = atlas_compounds_df['chromatography'].dropna().iloc[0]
ATLAS_POLARITY = atlas_compounds_df['polarity'].dropna().iloc[0]

print(f"Detected chromatography: {ATLAS_CHROMATOGRAPHY}")
print(f"Detected polarity: {ATLAS_POLARITY}")

# Verify all compounds have the same chromatography and polarity
unique_chromatography = atlas_compounds_df['chromatography'].dropna().unique()
unique_polarity = atlas_compounds_df['polarity'].dropna().unique()

if len(unique_chromatography) > 1:
    print(f"Warning: Multiple chromatography methods found: {unique_chromatography}")
    print(f"Using: {ATLAS_CHROMATOGRAPHY}")

if len(unique_polarity) > 1:
    print(f"Warning: Multiple polarities found: {unique_polarity}")
    print(f"Using: {ATLAS_POLARITY}")

# Get target adducts from input data
target_adducts = None
if 'adduct' in atlas_compounds_df.columns:
    unique_adducts = atlas_compounds_df['adduct'].dropna().unique().tolist()
    if unique_adducts:
        target_adducts = unique_adducts
        print(f"Target adducts: {target_adducts}")

print(f"\nAtlas will be created for {ATLAS_CHROMATOGRAPHY}/{ATLAS_POLARITY}")
if target_adducts:
    print(f"Will only include references with adducts: {target_adducts}")

Loading atlas compound list from: /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv
Loaded 20 compounds for atlas
Unique compounds for atlas: 20
Detected chromatography: HILICZ
Detected polarity: positive
Target adducts: ['[M+H]+']

Atlas will be created for HILICZ/positive
Will only include references with adducts: ['[M+H]+']


In [4]:
# Check if database exists
if not os.path.exists(DATABASE_PATH):
    raise FileNotFoundError(f"Database not found at {DATABASE_PATH}. Please run Add_To_Compounds_Database.ipynb first.")

print(f"Validating compounds exist in database: {DATABASE_PATH}")

# Connect to database and check for existing compounds
conn = duckdb.connect(str(DATABASE_PATH))

# Get all compounds currently in database
existing_compounds = conn.execute("SELECT inchi_key, compound_uid, name FROM compounds").fetchall()
existing_inchi_keys = {row[0]: {'compound_uid': row[1], 'name': row[2]} for row in existing_compounds}

print(f"Database contains {len(existing_inchi_keys)} compounds")

# Check which atlas compounds are missing from database
missing_compounds = []
available_compounds = []

for inchi_key in atlas_inchi_keys:
    if inchi_key in existing_inchi_keys:
        available_compounds.append({
            'inchi_key': inchi_key,
            'compound_uid': existing_inchi_keys[inchi_key]['compound_uid'],
            'name': existing_inchi_keys[inchi_key]['name']
        })
    else:
        # Get compound name from input data if available
        compound_name = atlas_compounds_df[atlas_compounds_df['inchi_key'] == inchi_key]['label'].iloc[0] if 'label' in atlas_compounds_df.columns else 'Unknown'
        missing_compounds.append({
            'inchi_key': inchi_key,
            'name': compound_name
        })

print(f"Compounds available for atlas: {len(available_compounds)}")
print(f"Compounds missing from database: {len(missing_compounds)}")

if missing_compounds:
    print("\nMissing compounds:")
    for compound in missing_compounds[:10]:  # Show first 10
        print(f"  - {compound['name']} ({compound['inchi_key']})")
    if len(missing_compounds) > 10:
        print(f"  ... and {len(missing_compounds) - 10} more")
    
    raise ValueError(f"Cannot create atlas: {len(missing_compounds)} compounds not found in database. "
                    f"Please add these compounds using Add_To_Compounds_Database.ipynb first.")

print(f"\nAll {len(available_compounds)} compounds found in database!")

# Get compound UIDs for atlas creation
compound_uids = [comp['compound_uid'] for comp in available_compounds]

conn.close()

Validating compounds exist in database: /Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb
Database contains 482 compounds
Compounds available for atlas: 20
Compounds missing from database: 0

All 20 compounds found in database!


## Create Atlas from Existing Compounds

In [8]:
# Create atlas from validated compounds
print(f"Creating new atlas: {ATLAS_NAME}")
print(f"Description: {ATLAS_DESC}")
print(f"Method: {ATLAS_CHROMATOGRAPHY}/{ATLAS_POLARITY}")
print(f"Compounds to include: {len(compound_uids)}")
print(f"Note: This will always create a new atlas with a unique UID, even if an atlas with the same name exists")

# Create the atlas (always creates new atlas with new UID)
atlas_uid, atlas_associations_created = dbi.create_atlas_from_compounds(
    DATABASE_PATH,
    ATLAS_NAME,
    ATLAS_DESC,
    ATLAS_CHROMATOGRAPHY,
    ATLAS_POLARITY,
    creator_name,
    compound_uids=compound_uids,
    target_adducts=target_adducts
)

print(f"\nAtlas creation complete!")
print(f"   Atlas Name: {ATLAS_NAME}")
print(f"   Atlas UID: {atlas_uid}")
print(f"   Atlas associations created: {atlas_associations_created}")
print(f"   Method: {ATLAS_CHROMATOGRAPHY}/{ATLAS_POLARITY}")

if atlas_associations_created != len(compound_uids):
    print(f"Warning: Compound associations to atlas did not match input atlas.")

# Validate the final database
dbi.validate_database(DATABASE_PATH)

Creating new atlas: HILIC Positive QC Default
Description: Default QC compounds for HILIC Positive as of 2025-08-27T09:27:43.188853
Method: HILICZ/positive
Compounds to include: 20
Note: This will always create a new atlas with a unique UID, even if an atlas with the same name exists
Created new atlas: HILIC Positive QC Default (UID: atlas-85b7fffb07e640e5a93e144618f11428)

Atlas creation complete!
   Atlas Name: HILIC Positive QC Default
   Atlas UID: atlas-85b7fffb07e640e5a93e144618f11428
   Atlas associations created: 20
   Method: HILICZ/positive

Database Validation:
   Compounds: 482
   RT/MZ References: 872
   Atlases: 3
   Atlas-Compound associations: 105
   Method combinations:
      HILICZ/negative: 466 references
      HILICZ/positive: 406 references
   Available atlases:
      atlas-fbceed0e6d8b4c349d8fcc81cf0fe9fd
            HILIC Positive ISTD Default
            HILICZ positive
            65 compounds
            2025-08-26 15:14:29.007115
      atlas-65c9223bd8be48cb8

In [ ]:
#dbi.delete_atlas_from_db(db_path=DATABASE_PATH, atlas_uid="atlas-85b7fffb07e640e5a93e144618f11428")

Found atlas: HILIC Positive QC Default
Description: Default QC compounds for HILIC Positive as of 2025-08-27T09:27:43.188853
Removed 20 compound associations

Successfully deleted atlas 'HILIC Positive QC Default' (UID: atlas-85b7fffb07e640e5a93e144618f11428)
Cascade deletions performed:
  - atlas_compound_associations: 20 records


'Deletion successful!'